In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("PremalMatalia/roberta-base-best-squad2")
model = AutoModelForQuestionAnswering.from_pretrained("PremalMatalia/roberta-base-best-squad2")

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/ubuntu/anaconda3/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [3]:
print(model)

RobertaForQuestionAnswering(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (Lay

In [4]:
#obfuscate model
model = model_obfuscation(model)
print(model)

Obfuscating: roberta.encoder.layer.0.attention.self.query
Obfuscating: roberta.encoder.layer.0.attention.self.key
Obfuscating: roberta.encoder.layer.0.attention.self.value
Obfuscating: roberta.encoder.layer.0.attention.output.dense
Obfuscating: roberta.encoder.layer.0.intermediate.dense
Obfuscating: roberta.encoder.layer.0.output.dense
Obfuscating: roberta.encoder.layer.1.attention.self.query
Obfuscating: roberta.encoder.layer.1.attention.self.key
Obfuscating: roberta.encoder.layer.1.attention.self.value
Obfuscating: roberta.encoder.layer.1.attention.output.dense
Obfuscating: roberta.encoder.layer.1.intermediate.dense
Obfuscating: roberta.encoder.layer.1.output.dense
Obfuscating: roberta.encoder.layer.2.attention.self.query
Obfuscating: roberta.encoder.layer.2.attention.self.key
Obfuscating: roberta.encoder.layer.2.attention.self.value
Obfuscating: roberta.encoder.layer.2.attention.output.dense
Obfuscating: roberta.encoder.layer.2.intermediate.dense
Obfuscating: roberta.encoder.layer.2

In [5]:
model = obfus_inference_mode(model)

In [6]:
question_answerer = pipeline("question-answering", model=model, tokenizer=tokenizer, device='cuda')

In [7]:
context = 'Tom likes playing with Jerry.'
question = 'Who does Tom like playing with?'
result = question_answerer(question=question, context=context)
print(result)

{'score': 0.02040819823741913, 'start': 4, 'end': 9, 'answer': 'likes'}


In [8]:
dataset = load_dataset("rajpurkar/squad")['validation']
print(dataset)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10570
})


In [9]:
correct = 0
total = 0

for i in tqdm.tqdm(range(10570)):
    result = question_answerer(question=dataset[i]['question'], context=dataset[i]['context'])
    for answer in dataset[i]['answers']['text']: # exact match
        if answer == result['answer']:
            correct += 1
            break
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 10570/10570 [01:42<00:00, 102.74it/s]

correct:68, total:10570, accuracy:0.006433301797540208
